# 📈 End-to-End Sales Forecasting & Demand Intelligence System
**Course Project — Week 3 & Week 4**  
**Intern:** Rishi Srivastava  
**Organization:** Xylofy AI  

---

## 📋 Project Overview
This project builds an intelligent demand forecasting, anomaly detection, and product segmentation system for a retail company. The core objective is to answer: **"How much of each product will we sell next month, and will we have enough stock to meet that demand?"**

We will:
1. Load, clean, and explore historical transaction data (`train.csv`) and industry-wide video game sales (`vgsales.csv`).
2. Perform seasonal decomposition and stationarity testing.
3. Compare three models: **SARIMA (Statistical)**, **Prophet (Industry standard)**, and **XGBoost (ML-based)**.
4. Forecast segment-level sales (Furniture, Technology, Office Supplies, West, East).
5. Detect weekly sales anomalies using **Isolation Forest** and **Z-Score** models.
6. Group product sub-categories into demand groups using **K-Means Clustering**.
7. Develop a Streamlit dashboard (`app.py`).
8. Generate a PDF report for the CFO and Head of Supply Chain (`summary.pdf`).


## 🔍 Task 1 — Data Loading, Merging & Deep Exploration
In this section, we load the Superstore sales dataset, parse datetimes, extract calendar features, impute missing values, and merge it with video game sales data to compare trends.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load data
df = pd.read_csv('train.csv', encoding='utf-8')
df_vg = pd.read_csv('vgsales.csv')

# Parse datetimes
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y', errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y', errors='coerce')

# Check and impute missing Postal Code (Burlington, Vermont postal code = 05401)
print("Postal code missing rows count:", df['Postal Code'].isnull().sum())
df.loc[df['City'] == 'Burlington', 'Postal Code'] = df.loc[df['City'] == 'Burlington', 'Postal Code'].fillna(5401.0)
df['Postal Code'] = df['Postal Code'].astype(int)

# Extract features
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Week'] = df['Order Date'].dt.isocalendar().week.astype(int)
df['DayOfWeek'] = df['Order Date'].dt.dayofweek
df['Quarter'] = df['Order Date'].dt.quarter
df['Season'] = df['Month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
})

print("Preprocessed Superstore dataset shape:", df.shape)


### Merging Analysis
We aggregate both Superstore Sales and Video Game Sales by Year and join them to analyze historical revenue patterns.


In [ ]:
# Clean vgsales
df_vg_cleaned = df_vg.dropna(subset=['Year']).copy()
df_vg_cleaned['Year'] = df_vg_cleaned['Year'].astype(int)
vg_yearly = df_vg_cleaned.groupby('Year')['Global_Sales'].sum().reset_index()

# Merge on Year
superstore_yearly = df.groupby('Year')['Sales'].sum().reset_index()
merged_yearly = pd.merge(superstore_yearly, vg_yearly, on='Year', how='inner')
print("Merged Yearly Sales Data:")
print(merged_yearly)


### Task 1 Questions & Answers
1. **Which product category generates the highest total revenue?**
2. **Which region has the most consistent sales growth over 4 years?**
3. **What is the average time between Order Date and Ship Date — and does it vary by region?**
4. **Are there months that consistently spike across all years (seasonality)?**


In [ ]:
# Q1: Highest Revenue Category
cat_rev = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print("Q1: Revenue by Category:")
print(cat_rev)
print(f"-> Highest Category: {cat_rev.index[0]} (${cat_rev.iloc[0]:,.2f})\n")

# Q2: Consistent Sales Growth by Region
region_yearly = df.groupby(['Region', 'Year'])['Sales'].sum().unstack()
region_growth = region_yearly.pct_change(axis=1) * 100
growth_vol = region_growth.std(axis=1)
print("Q2: Region Yearly Sales:")
print(region_yearly)
print("\nRegion YoY Growth (%):")
print(region_growth)
print("\nGrowth Rate Volatility (Std Dev):")
print(growth_vol)
print("-> East Region has the most consistent growth (positive every year, lowest std dev of 1.79%)\n")

# Q3: Average Shipping Time
df['ShipTime'] = (df['Ship Date'] - df['Order Date']).dt.days
print("Q3: Avg Shipping Days by Region:")
print(df.groupby('Region')['ShipTime'].mean())
print("Overall Mean Shipping Time:", df['ShipTime'].mean(), "days\n")

# Q4: Seasonality Monthly Spikes
monthly_sales = df.groupby(['Year', 'Month'])['Sales'].sum().unstack()
print("Q4: Monthly sales across years:")
print(monthly_sales)
print("-> November and December consistently show large spikes across all years.")


## 📈 Task 2 — Time Series Analysis & Decomposition
We resample the sales data to monthly totals, run seasonal decomposition, and check for stationarity using the Augmented Dickey-Fuller (ADF) test.


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

df_monthly_ts = df.set_index('Order Date').resample('ME')['Sales'].sum()

# Run decomposition
decomp = seasonal_decompose(df_monthly_ts, model='additive', period=12)

# Plot components
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
decomp.observed.plot(ax=axes[0], color='#2c3e50', lw=2)
axes[0].set_title('Observed Sales')
decomp.trend.plot(ax=axes[1], color='#e67e22', lw=2)
axes[1].set_title('Trend')
decomp.seasonal.plot(ax=axes[2], color='#2ecc71', lw=2)
axes[2].set_title('Seasonality')
decomp.resid.plot(ax=axes[3], color='#e74c3c', style='o')
axes[3].set_title('Residuals')
plt.tight_layout()
os.makedirs('charts', exist_ok=True)
plt.savefig('charts/decomposition.png', dpi=150)
plt.show()

# ADF test
def adf_test(series):
    result = adfuller(series)
    print(f"ADF Stat: {result[0]:.4f}, p-val: {result[1]:.4f}")
    return result[1]

p_val = adf_test(df_monthly_ts)
if p_val > 0.05:
    print("Series is non-stationary. Applying differencing...")
    df_diff = df_monthly_ts.diff().dropna()
    adf_test(df_diff)
else:
    print("Series is stationary.")


## 🔮 Task 3 — Sales Forecasting using 3 Different Models
We build, validate, and compare **SARIMA**, **Prophet**, and **XGBoost** models on monthly sales.


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from prophet import Prophet
import xgboost as xgb

# Train/Val Split
train_ts = df_monthly_ts[:-6]
val_ts = df_monthly_ts[-6:]

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

metrics = {}


### Model 1: SARIMA (Statistical Model)
We fit a SARIMA(1,1,1)x(1,1,1,12) model.


In [ ]:
sarima_model = SARIMAX(train_ts, order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)
sarima_val_pred = sarima_fit.predict(start=val_ts.index[0], end=val_ts.index[-1])

# Retrain on full series for forecasting
full_sarima = SARIMAX(df_monthly_ts, order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False)
full_sarima_fit = full_sarima.fit(disp=False)
sarima_forecast = full_sarima_fit.get_forecast(steps=3)
sarima_forecast_mean = sarima_forecast.predicted_mean
sarima_forecast_ci = sarima_forecast.conf_int()

metrics['SARIMA'] = {
    'MAE': mean_absolute_error(val_ts, sarima_val_pred),
    'RMSE': root_mean_squared_error(val_ts, sarima_val_pred),
    'MAPE': mape(val_ts, sarima_val_pred)
}


### Model 2: Facebook Prophet
We prepare the data in the required format and train Prophet.


In [ ]:
df_p_train = train_ts.reset_index().rename(columns={'Order Date': 'ds', 'Sales': 'y'})
df_p_train['ds'] = df_p_train['ds'].dt.tz_localize(None)

m_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m_prophet.fit(df_p_train)

# Predict validation
future_val = m_prophet.make_future_dataframe(periods=6, freq='ME')
prophet_pred_full = m_prophet.predict(future_val)
prophet_val_pred = prophet_pred_full['yhat'].iloc[-6:].values

# Forecast 3 months
df_p_full = df_monthly_ts.reset_index().rename(columns={'Order Date': 'ds', 'Sales': 'y'})
df_p_full['ds'] = df_p_full['ds'].dt.tz_localize(None)
m_prophet_full = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m_prophet_full.fit(df_p_full)
future_3m = m_prophet_full.make_future_dataframe(periods=3, freq='ME')
prophet_forecast_full = m_prophet_full.predict(future_3m)
prophet_forecast_mean = prophet_forecast_full['yhat'].iloc[-3:].values

# Plot components
fig = m_prophet_full.plot_components(prophet_forecast_full)
fig.savefig('charts/prophet_components.png', dpi=150)
plt.show()

metrics['Prophet'] = {
    'MAE': mean_absolute_error(val_ts, prophet_val_pred),
    'RMSE': root_mean_squared_error(val_ts, prophet_val_pred),
    'MAPE': mape(val_ts, prophet_val_pred)
}


### Model 3: XGBoost for Time Series
We engineer lag and calendar features, then train XGBoost.


In [ ]:
def create_xgb_features(series):
    df_feat = pd.DataFrame(series)
    df_feat.columns = ['Sales']
    df_feat['Lag1'] = df_feat['Sales'].shift(1)
    df_feat['Lag2'] = df_feat['Sales'].shift(2)
    df_feat['Lag3'] = df_feat['Sales'].shift(3)
    df_feat['RollingMean'] = df_feat['Sales'].shift(1).rolling(window=3).mean()
    df_feat['Month'] = df_feat.index.month
    df_feat['Quarter'] = df_feat.index.quarter
    df_feat = df_feat.bfill()
    return df_feat

df_xgb = create_xgb_features(df_monthly_ts)
X_xgb = df_xgb.drop(columns=['Sales'])
y_xgb = df_xgb['Sales']

X_train_xgb, X_val_xgb = X_xgb[:-6], X_xgb[-6:]
y_train_xgb, y_val_xgb = y_xgb[:-6], y_xgb[-6:]

xgb_reg = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
xgb_reg.fit(X_train_xgb, y_train_xgb)
xgb_val_pred = xgb_reg.predict(X_val_xgb)

# Forecast 3 months ahead autoregressively
xgb_reg_full = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
xgb_reg_full.fit(X_xgb, y_xgb)

last_known = df_xgb.copy()
forecast_dates = pd.date_range(start=df_monthly_ts.index[-1] + pd.Timedelta(days=1), periods=3, freq='ME')

xgb_forecast_mean = []
for f_date in forecast_dates:
    lag1 = last_known['Sales'].iloc[-1]
    lag2 = last_known['Sales'].iloc[-2]
    lag3 = last_known['Sales'].iloc[-3]
    roll_mean = last_known['Sales'].iloc[-3:].mean()
    m_val = f_date.month
    q_val = f_date.quarter
    
    X_f = pd.DataFrame([[lag1, lag2, lag3, roll_mean, m_val, q_val]], columns=X_xgb.columns)
    pred_val = xgb_reg_full.predict(X_f)[0]
    xgb_forecast_mean.append(pred_val)
    
    new_row = pd.DataFrame([[pred_val, lag1, lag2, lag3, roll_mean, m_val, q_val]], columns=['Sales'] + list(X_xgb.columns), index=[f_date])
    last_known = pd.concat([last_known, new_row])

metrics['XGBoost'] = {
    'MAE': mean_absolute_error(val_ts, xgb_val_pred),
    'RMSE': root_mean_squared_error(val_ts, xgb_val_pred),
    'MAPE': mape(val_ts, xgb_val_pred)
}


### Model Comparison & Validation Plots
We compare model performance metrics.


In [ ]:
# Print Comparison Table
comp_df = pd.DataFrame(metrics).T
comp_df['Forecast Month 1'] = [sarima_forecast_mean.iloc[0], prophet_forecast_mean[0], xgb_forecast_mean[0]]
comp_df['Forecast Month 2'] = [sarima_forecast_mean.iloc[1], prophet_forecast_mean[1], xgb_forecast_mean[1]]
comp_df['Forecast Month 3'] = [sarima_forecast_mean.iloc[2], prophet_forecast_mean[2], xgb_forecast_mean[2]]
print(comp_df)

# Plot comparison
plt.figure(figsize=(10, 5))
plt.plot(df_monthly_ts.index[-18:], df_monthly_ts[-18:], label='Actual', color='#2c3e50', marker='o')
plt.plot(val_ts.index, sarima_val_pred, label='SARIMA', color='#e67e22', linestyle='--', marker='s')
plt.plot(val_ts.index, prophet_val_pred, label='Prophet', color='#2ecc71', linestyle='--', marker='^')
plt.plot(val_ts.index, xgb_val_pred, label='XGBoost', color='#9b59b6', linestyle='--', marker='d')
plt.title('Validation Comparison')
plt.legend()
plt.savefig('charts/validation_comparison.png', dpi=150)
plt.show()


## 🔮 Task 4 — Product Category & Region Level Forecasting
We run the best-performing model (SARIMA) separately on each segment: Furniture, Technology, Office Supplies, West Region, East Region.


In [ ]:
segments = {
    'Furniture': df[df['Category'] == 'Furniture'],
    'Technology': df[df['Category'] == 'Technology'],
    'Office Supplies': df[df['Category'] == 'Office Supplies'],
    'West Region': df[df['Region'] == 'West'],
    'East Region': df[df['Region'] == 'East']
}

segment_forecasts = {}
plt.figure(figsize=(12, 6))
for name, seg_df in segments.items():
    seg_ts = seg_df.set_index('Order Date').resample('ME')['Sales'].sum()
    model = SARIMAX(seg_ts, order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    forecast = fit.predict(start=len(seg_ts), end=len(seg_ts)+2)
    forecast.index = pd.date_range(start='2019-01-31', periods=3, freq='ME')
    segment_forecasts[name] = forecast
    plt.plot(forecast.index, forecast.values, label=name, marker='o')

plt.title('3-Month Segment Forecasts')
plt.ylabel('Sales ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('charts/segment_forecasts.png', dpi=150)
plt.show()


## ⚠️ Task 5 — Anomaly Detection in Sales Data
We apply **Isolation Forest** and **Z-Score** models to identify weekly sales outliers.


In [ ]:
from sklearn.ensemble import IsolationForest

weekly_sales = df.set_index('Order Date').resample('W')['Sales'].sum().reset_index()

# 1. Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
weekly_sales['Anomaly_IF'] = iso_forest.fit_predict(weekly_sales[['Sales']])
weekly_sales['Anomaly_IF'] = weekly_sales['Anomaly_IF'].map({1: 0, -1: 1})

# 2. Z-Score (12-week rolling window)
weekly_sales['Rolling_Mean'] = weekly_sales['Sales'].rolling(window=12, min_periods=1).mean()
weekly_sales['Rolling_Std'] = weekly_sales['Sales'].rolling(window=12, min_periods=1).std().fillna(weekly_sales['Sales'].std())
weekly_sales['Z_Score'] = (weekly_sales['Sales'] - weekly_sales['Rolling_Mean']) / weekly_sales['Rolling_Std']
weekly_sales['Anomaly_Z'] = (np.abs(weekly_sales['Z_Score']) > 2.0).astype(int)

# Plot Isolation Forest
plt.figure(figsize=(14, 5))
plt.plot(weekly_sales['Order Date'], weekly_sales['Sales'], label='Weekly Sales', color='#94a3b8')
anoms_if = weekly_sales[weekly_sales['Anomaly_IF'] == 1]
plt.scatter(anoms_if['Order Date'], anoms_if['Sales'], color='#ef4444', label='Outlier')
plt.title('Weekly Sales Anomalies (Isolation Forest)')
plt.legend()
plt.savefig('charts/anomalies_isolation_forest.png', dpi=150)
plt.show()

# Video game sales outliers
vg_outliers = df_vg[df_vg['Global_Sales'] > df_vg['Global_Sales'].quantile(0.99)]
print("Top Video Game Outliers:")
print(vg_outliers[['Name', 'Platform', 'Year', 'Global_Sales']].head(3))


## 🧩 Task 6 — Product Demand Segmentation using Clustering
We group product sub-categories using K-Means Clustering on sales volume, growth, volatility, and average order value.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Aggregate features
sub_cat_rev = df.groupby('Sub-Category')['Sales'].sum().reset_index()
sub_cat_volume = df.groupby('Sub-Category')['Order ID'].nunique().reset_index().rename(columns={'Order ID': 'Volume'})
sub_cat_monthly = df.groupby(['Sub-Category', 'Year', 'Month'])['Sales'].sum().reset_index()
sub_cat_volatility = sub_cat_monthly.groupby('Sub-Category')['Sales'].std().reset_index().rename(columns={'Sales': 'Volatility'})
sub_cat_yearly = df.groupby(['Sub-Category', 'Year'])['Sales'].sum().unstack().fillna(0)
sub_cat_growth = ((sub_cat_yearly[2018] - sub_cat_yearly[2015]) / (sub_cat_yearly[2015] + 1)).reset_index().rename(columns={0: 'Growth'})
sub_order_value = df.groupby('Sub-Category')['Sales'].mean().reset_index().rename(columns={'Sales': 'AvgOrderValue'})

features = sub_cat_rev.merge(sub_cat_volume, on='Sub-Category')\
                      .merge(sub_cat_volatility, on='Sub-Category')\
                      .merge(sub_cat_growth, on='Sub-Category')\
                      .merge(sub_order_value, on='Sub-Category')

scaler = StandardScaler()
X_clust = scaler.fit_transform(features.drop(columns=['Sub-Category']))

# Elbow Curve
wcss = []
for k in range(1, 10):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_clust)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 10), wcss, marker='o')
plt.title('Elbow Curve')
plt.savefig('charts/elbow_curve.png', dpi=150)
plt.show()

# K-Means clustering (k=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
features['Cluster'] = kmeans.fit_predict(X_clust)

# Reduce to 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clust)
features['PCA1'] = X_pca[:, 0]
features['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=features, x='PCA1', y='PCA2', hue='Cluster', palette='Set1', s=100)
for i, row in features.iterrows():
    plt.text(row['PCA1'] + 0.1, row['PCA2'] + 0.1, row['Sub-Category'], fontsize=8)
plt.title('Sub-Category Clusters')
plt.savefig('charts/kmeans_pca_clusters.png', dpi=150)
plt.show()
